In [1]:
import pandas as pd
import plotly.graph_objects as go
from tradeo.ohlc import OHLC
from datetime import timedelta
from pandas.tseries.offsets import BDay
import math
import numpy as np
import time
from IPython.display import clear_output

from tradeo.trading_methods import calculate_heikin_ashi
from sorul_tradingbot.strategy.private.volume_16 import Volume
from sorul_tradingbot.strategy.simulator.simulator import SimulatedMTClient
from sorul_tradingbot.strategy.simulator.simulator import build_simulation_heikin_ashi


# Funciones

In [2]:
import math
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from pandas.tseries.offsets import BDay

from sorul_tradingbot.strategy.private.volume_19 import Volume
from sorul_tradingbot.strategy.simulator.simulator import SimulatedMTClient


def normalize_levels_period(levels_period: str | None) -> str | None:
  if levels_period is None:
    return None
  levels_period = levels_period.strip().lower()
  if levels_period not in ("daily", "weekly"):
    raise ValueError("levels_period must be 'daily', 'weekly', or None")
  return levels_period


def to_utc_timestamp(value) -> pd.Timestamp:
  ts = pd.Timestamp(value)
  if ts.tzinfo is None:
    return ts.tz_localize("UTC")
  return ts.tz_convert("UTC")


def get_plot_window(
    date: str,
    levels_period: str = "weekly",
    context_bdays: int | None = None,
    extra_calendar_days: int | None = None,
):
  levels_period = normalize_levels_period(levels_period) or "weekly"

  if context_bdays is None:
    context_bdays = 2 if levels_period == "daily" else 7

  if extra_calendar_days is None:
    extra_calendar_days = 2 if levels_period == "daily" else 7

  center = to_utc_timestamp(date)
  start_dt = (center - BDay(context_bdays)).normalize()
  end_dt = (center + BDay(context_bdays)).normalize()
  end_dt = end_dt + pd.Timedelta(days=extra_calendar_days)

  return start_dt, end_dt


# ---- 1) Carga de payload + parsing de fechas ----
def load_payload(
    path: str,
    levels_period: str | None = None,
):
  """
  levels_period:
    - "daily"  -> carga solo niveles diarios
    - "weekly" -> carga solo niveles semanales
    - None     -> no filtra por periodo
  """
  levels_period = normalize_levels_period(levels_period)

  payload = pd.read_pickle(path)

  candles = payload["candles"].copy()
  candles["datetime"] = pd.to_datetime(candles["datetime"])
  candles = candles.set_index("datetime")

  closed_trades = payload["closed_trades"].copy()
  if not closed_trades.empty:
    closed_trades["entry_time"] = pd.to_datetime(
        closed_trades["entry_time"], utc=True, errors="coerce"
    )
    closed_trades["exit_time"] = pd.to_datetime(
        closed_trades["exit_time"], utc=True, errors="coerce"
    )

  levels = payload["levels"].copy()

  if not levels.empty:
    levels["timestamp"] = pd.to_datetime(
        levels["timestamp"], utc=True, errors="coerce"
    )
    levels["levels_date"] = pd.to_datetime(
        levels["levels_date"], errors="coerce"
    )

    # Compatibilidad con payloads antiguos: antes todo era semanal.
    if "levels_period" not in levels.columns:
      levels["levels_period"] = "weekly"

    levels["levels_period"] = (
        levels["levels_period"]
        .fillna("weekly")
        .astype(str)
        .str.strip()
        .str.lower()
    )

    if levels_period is not None:
      levels = levels[levels["levels_period"] == levels_period].copy()

    levels["plot_date"] = levels["timestamp"].dt.normalize()

    # Importante: no mezclar diarios y semanales.
    levels = levels.drop_duplicates(
        subset=["levels_period", "plot_date"],
        keep="first",
    )
  else:
    levels["plot_date"] = pd.NaT
    levels["levels_period"] = levels_period or "weekly"

  closed_trades = convert_sp500_closed_trades_pnl_to_eur(closed_trades)
  return candles, closed_trades, levels


# ---- 2) Tagging ----
def add_tag_column(closed_trades: pd.DataFrame) -> pd.DataFrame:
  if closed_trades.empty:
    return closed_trades
  out = closed_trades.copy()
  out["tag"] = out["comment"].apply(
      lambda x: x.split("--")[1] if "--" in x else ""
  )
  return out


# ---- 3) Filtro por tag opcional ----
def filter_trades_by_tag(
    trades: pd.DataFrame,
    tag: str | None,
) -> pd.DataFrame:
  if tag is None or trades.empty:
    return trades
  return trades[trades["tag"] == tag].copy()


# ---- 4) Helpers de tiempo ----
def ensure_utc_index(df: pd.DataFrame) -> pd.DataFrame:
  idx = pd.to_datetime(df.index)
  if getattr(idx, "tz", None) is None:
    idx = idx.tz_localize("UTC")
  else:
    idx = idx.tz_convert("UTC")
  out = df.copy()
  out.index = idx
  return out


# ---- 5) Plot ----
def plot_candles_and_trades(
    candles: pd.DataFrame,
    trades: pd.DataFrame | None = None,
    date: str | None = None,
    levels_period: str = "weekly",
    context_bdays: int | None = None,
    extra_calendar_days: int | None = None,
):
  cdl = ensure_utc_index(candles)

  if date is not None:
    start_dt, end_dt = get_plot_window(
        date=date,
        levels_period=levels_period,
        context_bdays=context_bdays,
        extra_calendar_days=extra_calendar_days,
    )

    cdl = cdl[(cdl.index >= start_dt) & (cdl.index < end_dt)]

    if trades is not None and not trades.empty:
      trades = trades.copy()
      trades["entry_time"] = pd.to_datetime(
          trades["entry_time"], utc=True, errors="coerce"
      )
      trades["exit_time"] = pd.to_datetime(
          trades["exit_time"], utc=True, errors="coerce"
      )
      mask = (
          (trades["entry_time"] >= start_dt)
          & (trades["entry_time"] < end_dt)
      )
      trades = trades[mask]

  fig = go.Figure()
  fig.add_trace(
      go.Candlestick(
          x=cdl.index,
          open=cdl["open"],
          high=cdl["high"],
          low=cdl["low"],
          close=cdl["close"],
          name="OHLC",
      )
  )

  if trades is not None and not trades.empty:
    trs = trades.copy()
    trs["entry_time"] = pd.to_datetime(
        trs["entry_time"], utc=True, errors="coerce"
    )
    trs["exit_time"] = pd.to_datetime(
        trs["exit_time"], utc=True, errors="coerce"
    )

    for _, r in trs.iterrows():
      et = r["entry_time"]
      xt = r["exit_time"]
      ep = float(r["entry_price"])
      xp = float(r["exit_price"])
      tp = float(r["take_profit"]) if pd.notna(r.get("take_profit")) else None
      sl = float(r["stop_loss"]) if pd.notna(r.get("stop_loss")) else None

      fig.add_trace(
          go.Scatter(
              x=[et],
              y=[ep],
              mode="markers",
              name="Entry",
              marker_symbol="triangle-up",
              marker_size=10,
              hovertext=[f'Entry {r.get("tag", "")}'],
              hoverinfo="text",
          )
      )
      fig.add_trace(
          go.Scatter(
              x=[xt],
              y=[xp],
              mode="markers",
              name="Exit",
              marker_symbol="triangle-down",
              marker_size=10,
              hovertext=[f'Exit ({r.get("result", "")})'],
              hoverinfo="text",
          )
      )
      fig.add_trace(
          go.Scatter(
              x=[et, xt],
              y=[ep, xp],
              mode="lines",
              name="Trade",
              hoverinfo="skip",
          )
      )

      if tp is not None:
        fig.add_trace(
            go.Scatter(
                x=[et, xt],
                y=[tp, tp],
                mode="lines",
                name="TP",
                line=dict(dash="dash"),
                hoverinfo="text",
                hovertext=[f"TP: {tp}", f"TP: {tp}"],
            )
        )

      if sl is not None:
        fig.add_trace(
            go.Scatter(
                x=[et, xt],
                y=[sl, sl],
                mode="lines",
                name="SL",
                line=dict(dash="dot"),
                hoverinfo="text",
                hovertext=[f"SL: {sl}", f"SL: {sl}"],
            )
        )

  fig.update_layout(
      xaxis_rangeslider_visible=False,
      hovermode="x unified",
      legend_title_text="Legend",
      height=700,
      dragmode="zoom",
  )
  fig.update_xaxes(rangebreaks=[dict(bounds=["sat", "mon"])])
  fig.update_yaxes(fixedrange=False)

  return fig


def overlay_levels(
    fig,
    candles: pd.DataFrame,
    levels: pd.DataFrame,
    date: str | None = None,
    levels_period: str | None = None,
    context_bdays: int | None = None,
    extra_calendar_days: int | None = None,
):
  """
  levels_period:
    - "daily"  -> pinta niveles diarios
    - "weekly" -> pinta niveles semanales
    - None     -> usa los niveles que vengan en el DataFrame
  """
  levels_period = normalize_levels_period(levels_period)

  if levels.empty:
    return fig

  candles_utc = ensure_utc_index(candles)
  levels = levels.copy()

  levels["timestamp"] = pd.to_datetime(
      levels["timestamp"], utc=True, errors="coerce"
  )
  levels["levels_date"] = pd.to_datetime(
      levels["levels_date"], errors="coerce"
  )

  if "levels_period" not in levels.columns:
    levels["levels_period"] = "weekly"

  levels["levels_period"] = (
      levels["levels_period"]
      .fillna("weekly")
      .astype(str)
      .str.strip()
      .str.lower()
  )

  if levels_period is not None:
    levels = levels[levels["levels_period"] == levels_period].copy()

  if levels.empty:
    return fig

  if date is not None:
    period_for_window = levels_period or levels["levels_period"].iloc[0]
    start_dt, end_dt = get_plot_window(
        date=date,
        levels_period=period_for_window,
        context_bdays=context_bdays,
        extra_calendar_days=extra_calendar_days,
    )

    levels = levels[
        (levels["timestamp"] >= start_dt)
        & (levels["timestamp"] < end_dt)
    ].copy()

  if levels.empty:
    return fig

  levels["plot_date"] = levels["timestamp"].dt.normalize()

  # Para calcular horario de sesión con la misma lógica de la estrategia.
  # El levels_period no cambia _get_session_times, pero se pasa por coherencia.
  period_for_strategy = levels_period or levels["levels_period"].iloc[0]
  vol = Volume(
      SimulatedMTClient(),
      levels_period=period_for_strategy,
  )

  vah_first, poc_first, val_first = True, True, True

  for _, row in levels.iterrows():
    plot_date = row["plot_date"]
    if plot_date.tzinfo is None:
      plot_date = plot_date.tz_localize("UTC")

    start_t, end_t = vol._get_session_times(plot_date.to_pydatetime())

    start_dt = plot_date.replace(
        hour=start_t.hour,
        minute=start_t.minute,
        second=0,
        microsecond=0,
    )
    end_dt = plot_date.replace(
        hour=end_t.hour,
        minute=end_t.minute,
        second=0,
        microsecond=0,
    )

    if end_dt <= start_dt:
      end_dt += pd.Timedelta(days=1)

    mask = (candles_utc.index >= start_dt) & (candles_utc.index <= end_dt)
    if not mask.any():
      continue

    x0, x1 = candles_utc.index[mask][[0, -1]]
    row_period = row.get("levels_period", period_for_strategy)
    levels_date = row.get("levels_date", "")

    fig.add_trace(
        go.Scatter(
            x=[x0, x1],
            y=[row["VAH"], row["VAH"]],
            mode="lines",
            line=dict(color="#1f3c88", width=2),
            name=f"VAH {row_period}",
            legendgroup=f"VAH-{row_period}",
            showlegend=vah_first,
            hoverinfo="text",
            hovertext=[
                f"VAH {row_period}: {row['VAH']}<br>"
                f"levels_date: {levels_date}"
            ] * 2,
        )
    )
    fig.add_trace(
        go.Scatter(
            x=[x0, x1],
            y=[row["POC"], row["POC"]],
            mode="lines",
            line=dict(color="#881f1f", width=2),
            name=f"POC {row_period}",
            legendgroup=f"POC-{row_period}",
            showlegend=poc_first,
            hoverinfo="text",
            hovertext=[
                f"POC {row_period}: {row['POC']}<br>"
                f"levels_date: {levels_date}"
            ] * 2,
        )
    )
    fig.add_trace(
        go.Scatter(
            x=[x0, x1],
            y=[row["VAL"], row["VAL"]],
            mode="lines",
            line=dict(color="#6ecff6", width=2),
            name=f"VAL {row_period}",
            legendgroup=f"VAL-{row_period}",
            showlegend=val_first,
            hoverinfo="text",
            hovertext=[
                f"VAL {row_period}: {row['VAL']}<br>"
                f"levels_date: {levels_date}"
            ] * 2,
        )
    )

    vah_first, poc_first, val_first = False, False, False

  return fig


def tag_profitability(df: pd.DataFrame) -> pd.DataFrame:
  break_even_threshold = 0.1
  z_score = 1.96

  if df.empty:
    return pd.DataFrame()

  df = df.copy()
  df["pnl"] = pd.to_numeric(df["pnl"], errors="coerce")
  df = df.dropna(subset=["pnl"])

  wins_mask = df["pnl"] > break_even_threshold
  losses_mask = df["pnl"] < -break_even_threshold
  break_even_mask = ~(wins_mask | losses_mask)

  ganados = int(wins_mask.sum())
  perdidos = int(losses_mask.sum())
  break_evens = int(break_even_mask.sum())

  print("ganados", ganados)
  print("perdidos", perdidos)
  print("break_evens", break_evens)

  denominator = ganados + perdidos
  ratio = round(100 * ganados / denominator, 2) if denominator else 0
  print("ratio", ratio, "%")
  print("pnl total:", df["pnl"].sum())

  rows = []

  for tag, tag_df in df.groupby("tag", sort=False):
    tag_df = tag_df.sort_index()
    pnl = tag_df["pnl"]

    wins = pnl[pnl > break_even_threshold]
    losses = pnl[pnl < -break_even_threshold]
    break_evens = pnl[
        (pnl >= -break_even_threshold)
        & (pnl <= break_even_threshold)
    ]

    n_trades = len(pnl)
    n_wins = len(wins)
    n_losses = len(losses)
    n_break_even = len(break_evens)
    n_decisive = n_wins + n_losses

    win_rate = n_wins / n_decisive if n_decisive else 0
    loss_rate = n_losses / n_decisive if n_decisive else 0
    break_even_rate = n_break_even / n_trades if n_trades else 0

    avg_win = wins.mean() if n_wins else 0
    avg_loss = abs(losses.mean()) if n_losses else 0

    payoff_ratio = (
        avg_win / avg_loss
        if avg_loss
        else np.inf if avg_win else 0
    )

    gross_profit = wins.sum()
    gross_loss = abs(losses.sum())

    profit_factor = (
        gross_profit / gross_loss
        if gross_loss
        else np.inf if gross_profit else 0
    )

    expectancy = pnl.mean()
    pnl_std = pnl.std()

    sharpe = (
        expectancy / pnl_std
        if pd.notna(pnl_std) and pnl_std > 0
        else 0
    )

    cumulative_pnl = pnl.cumsum()
    drawdown = cumulative_pnl - cumulative_pnl.cummax()
    max_drawdown = abs(drawdown.min()) if not drawdown.empty else 0

    # Intervalo de confianza Wilson del 95% para el win rate.
    if n_decisive:
      denominator_ci = 1 + z_score**2 / n_decisive
      centre = (
          win_rate + z_score**2 / (2 * n_decisive)
      ) / denominator_ci
      margin = (
          z_score
          * np.sqrt(
              win_rate * (1 - win_rate) / n_decisive
              + z_score**2 / (4 * n_decisive**2)
          )
          / denominator_ci
      )
      win_rate_ci_low = max(0, centre - margin)
      win_rate_ci_high = min(1, centre + margin)
    else:
      win_rate_ci_low = 0
      win_rate_ci_high = 0

    # Cuánto depende el beneficio de las tres mejores operaciones.
    top_3_profit = wins.nlargest(3).sum() if n_wins else 0
    top_3_profit_share = (
        top_3_profit / gross_profit
        if gross_profit > 0
        else 0
    )

    rows.append({
        "tag": tag,
        "n_trades": n_trades,
        "n_wins": n_wins,
        "n_losses": n_losses,
        "n_break_even": n_break_even,
        "win_rate": win_rate,
        "win_rate_ci_low": win_rate_ci_low,
        "win_rate_ci_high": win_rate_ci_high,
        "loss_rate": loss_rate,
        "break_even_rate": break_even_rate,
        "pnl_sum": pnl.sum(),
        "pnl_mean": expectancy,
        "pnl_median": pnl.median(),
        "pnl_std": pnl_std,
        "pnl_q25": pnl.quantile(0.25),
        "pnl_q75": pnl.quantile(0.75),
        "best_trade": pnl.max(),
        "worst_trade": pnl.min(),
        "avg_win": avg_win,
        "avg_loss": avg_loss,
        "payoff_ratio": payoff_ratio,
        "expectancy": expectancy,
        "profit_factor": profit_factor,
        "max_drawdown": max_drawdown,
        "top_3_profit_share": top_3_profit_share,
        "sharpe_per_trade": sharpe,
    })

  return (
      pd.DataFrame(rows)
      .set_index("tag")
      .sort_values("expectancy", ascending=False)
  )


def filter_trades_by_result(
    trades: pd.DataFrame,
    result: str | None,
) -> pd.DataFrame:
  """
  result:
    - "win"  -> pnl > 0
    - "loss" -> pnl < 0
    - None   -> no filtra
  """
  if result is None or trades.empty:
    return trades

  if result == "win":
    return trades[trades["pnl"] > 0].copy()

  if result == "loss":
    return trades[trades["pnl"] < 0].copy()

  raise ValueError("result must be 'win', 'loss', or None")


def transformarExcel(
    ruta: str,
    horas_restadas: int,
    version: str,
) -> pd.DataFrame:
  posiciones = pd.read_excel(ruta, sheet_name="positions")
  ordenes = pd.read_excel(ruta, sheet_name="orders")[["Order", "Comment"]]

  df = posiciones.merge(
      ordenes,
      left_on="Position",
      right_on="Order",
      how="left",
  )

  if any(df["Comment"].isnull()):
    raise Exception("Hay alguna posición que no tiene comentario")

  df = df.rename(
      columns={
          "Position": "Ticket",
          "Time": "entry_time",
          "Time.1": "exit_time",
          "Commission": "Commission",
          "Symbol": "par",
          "Type": "type",
          "Profit": "profit",
          "Price": "entry_price",
          "Price.1": "exit_price",
          "Comment": "tag",
      }
  )

  df["result"] = np.where(
      df["profit"] > 0.2,
      "win",
      np.where(df["profit"] < -0.2, "lose", "break even"),
  )

  df["tag"] = df["tag"].str.strip()
  df["Commission"] = df["Commission"].astype("float")
  df["par"] = df.par.str.upper()

  df["entry_time"] = (
      pd.to_datetime(
          df["entry_time"],
          format="%Y.%m.%d %H:%M:%S",
          utc=True,
      )
      - timedelta(hours=horas_restadas)
  )
  df["exit_time"] = (
      pd.to_datetime(
          df["exit_time"],
          format="%Y.%m.%d %H:%M:%S",
          utc=True,
      )
      - timedelta(hours=horas_restadas)
  )

  df["hora"] = list(map(lambda x: x.hour, df["entry_time"]))
  df["minuto"] = list(map(lambda x: math.floor(x.minute / 5) * 5, df["entry_time"]))
  df["dia_semana"] = list(map(lambda x: x.weekday(), df["entry_time"]))

  # Filtramos
  df = df[df["tag"].str.contains(version, na=False)]

  return df


def convert_sp500_closed_trades_pnl_to_eur(
    closed_trades: pd.DataFrame,
) -> pd.DataFrame:
  """Devuelve el mismo DataFrame pero con `pnl` convertido a EUR.

  Replica la lógica del simulador:
  - toma el año de `exit_time`
  - descarga la media anual de EUR/USD
  - reemplaza `pnl` por `pnl / media_anual_eurusd`
  """
  import yfinance as yf

  if closed_trades.empty:
    return closed_trades.copy()

  required_columns = {"exit_time", "pnl"}
  missing = required_columns - set(closed_trades.columns)
  if missing:
    raise ValueError(f"Faltan columnas requeridas: {sorted(missing)}")

  out = closed_trades.copy()
  out["exit_time"] = pd.to_datetime(out["exit_time"], utc=True, errors="coerce")
  out["pnl"] = pd.to_numeric(out["pnl"], errors="coerce")

  if out["exit_time"].isna().any():
    raise ValueError("Hay filas con `exit_time` no valido.")

  if out["pnl"].isna().any():
    raise ValueError("Hay filas con `pnl` no valido.")

  yearly_rates: dict[int, float] = {}

  def fetch_average_eurusd_rate(year: int) -> float:
    start = datetime(year, 1, 1)
    end = datetime(year + 1, 1, 1)

    data = yf.download(
        "EURUSD=X",
        start=start,
        end=end,
        progress=False,
        auto_adjust=False,
    )

    closes = data.get("Close")
    if closes is None:
      raise ValueError("No se encontro la columna Close en EUR/USD.")

    closes = closes.dropna()
    if closes.empty:
      raise ValueError(f"No hay datos EUR/USD para el año {year}.")

    mean_value = closes.mean()
    if isinstance(mean_value, pd.Series):
      if mean_value.empty:
        raise ValueError(f"No hay datos EUR/USD para el año {year}.")
      mean_value = mean_value.iloc[0]

    return float(mean_value)

  years = out["exit_time"].dt.year.unique()
  for year in years:
    yearly_rates[int(year)] = fetch_average_eurusd_rate(int(year))

  out["pnl"] = out.apply(
      lambda row: float(row["pnl"] / yearly_rates[row["exit_time"].year]),
      axis=1,
  )

  return out

# Ejecuciones

In [ ]:
# 1. Load payload and report raw point performance
from pathlib import Path

import numpy as np
import pandas as pd

# Change only this path when analysing another simulator run.
payload_path = Path(
    "../simulator/outputs/"
    "pivot_trend_2026-09-21_22-02/"
    "simulation_payload_pivot_trend.pkl"
)

payload = pd.read_pickle(payload_path)

candles = payload["candles"].copy()
candles["datetime"] = pd.to_datetime(candles["datetime"], utc=True)
candles = candles.set_index("datetime").sort_index()

trades = payload["closed_trades"].copy()
trades["entry_time"] = pd.to_datetime(
    trades["entry_time"], utc=True, errors="coerce"
)
trades["exit_time"] = pd.to_datetime(
    trades["exit_time"], utc=True, errors="coerce"
)
trades["pnl_points"] = pd.to_numeric(trades["pnl"], errors="coerce")
trades = (
    trades.dropna(subset=["entry_time", "exit_time", "pnl_points"])
    .sort_values(["exit_time", "ticket"])
    .reset_index(drop=True)
)

break_even_threshold = 0.1
trades["holding_hours"] = (
    trades["exit_time"] - trades["entry_time"]
).dt.total_seconds() / 3600
trades["exit_year"] = trades["exit_time"].dt.year
trades["exit_month"] = trades["exit_time"].dt.strftime("%Y-%m")


def performance_metrics(frame: pd.DataFrame) -> pd.Series:
    pnl = frame["pnl_points"]
    wins = pnl[pnl > break_even_threshold]
    losses = pnl[pnl < -break_even_threshold]
    break_evens = pnl[
        (pnl >= -break_even_threshold)
        & (pnl <= break_even_threshold)
    ]

    equity = np.r_[0.0, pnl.cumsum().to_numpy()]
    drawdown = np.maximum.accumulate(equity) - equity

    return pd.Series(
        {
            "trades": len(frame),
            "net_points": pnl.sum(),
            "wins": len(wins),
            "losses": len(losses),
            "break_evens": len(break_evens),
            "win_rate_pct": (
                100 * len(wins) / (len(wins) + len(losses))
                if len(wins) + len(losses)
                else 0
            ),
            "profit_factor": (
                wins.sum() / abs(losses.sum()) if not losses.empty else np.inf
            ),
            "expectancy_points": pnl.mean(),
            "avg_win_points": wins.mean() if not wins.empty else 0,
            "avg_loss_points": losses.mean() if not losses.empty else 0,
            "max_closed_drawdown_points": drawdown.max(),
            "avg_holding_hours": frame["holding_hours"].mean(),
            "median_holding_hours": frame["holding_hours"].median(),
        }
    )


yearly = (
    trades.groupby("exit_year", group_keys=False)
    .apply(performance_metrics, include_groups=False)
    .reset_index()
    .round(2)
)
monthly_pnl = (
    trades.groupby("exit_month", as_index=False)["pnl_points"]
    .agg(trades="count", net_points="sum")
    .round(2)
)
by_side = (
    trades.groupby("side", group_keys=False)
    .apply(performance_metrics, include_groups=False)
    .reset_index()
    .round(2)
)
top_trades = (
    trades.nlargest(10, "pnl_points")[
        [
            "ticket",
            "side",
            "entry_time",
            "exit_time",
            "holding_hours",
            "pnl_points",
        ]
    ]
    .round({"holding_hours": 2, "pnl_points": 2})
)

best_trade = trades["pnl_points"].max()
net_points = trades["pnl_points"].sum()
net_without_best = net_points - best_trade

print("YEARLY PERFORMANCE — POINTS")
print(yearly.to_markdown(index=False))
print("\nPERFORMANCE BY SIDE — POINTS")
print(by_side.to_markdown(index=False))
print("\nMONTHLY P&L — POINTS")
print(monthly_pnl.to_markdown(index=False))
print("\nTOP 10 WINNERS — POINTS")
print(top_trades.to_markdown(index=False))
print(f"\nNet points: {net_points:.2f}")
print(f"Best trade: {best_trade:.2f}")
print(f"Net without best trade: {net_without_best:.2f}")


In [ ]:
# 2. Build causal market regimes at each entry
# All features are shifted to completed bars before an entry can use them.
previous_close = candles["close"].shift(1)
true_range = pd.concat(
    [
        candles["high"] - candles["low"],
        (candles["high"] - previous_close).abs(),
        (candles["low"] - previous_close).abs(),
    ],
    axis=1,
).max(axis=1)

atr_period = 14
trend_bars = 48  # Four M5 hours.

features = pd.DataFrame(index=candles.index)
features["atr_points"] = true_range.rolling(atr_period).mean().shift(1)
features["atr_pct"] = features["atr_points"] / candles["close"].shift(1)

price_change = candles["close"].diff()
features["efficiency"] = (
    candles["close"].sub(candles["close"].shift(trend_bars)).abs()
    / price_change.abs().rolling(trend_bars).sum()
).shift(1)
features["trend_move_atr"] = (
    candles["close"].sub(candles["close"].shift(trend_bars))
    / features["atr_points"]
).shift(1)
features["channel_width_atr"] = (
    candles["high"].rolling(trend_bars).max()
    - candles["low"].rolling(trend_bars).min()
) / features["atr_points"]
features["channel_width_atr"] = features["channel_width_atr"].shift(1)

analysis_trades = trades.join(features, on="entry_time", how="left")
analysis_trades = analysis_trades.dropna(
    subset=[
        "atr_points",
        "atr_pct",
        "efficiency",
        "trend_move_atr",
        "channel_width_atr",
    ]
).copy()
analysis_trades["entry_year"] = analysis_trades["entry_time"].dt.year
analysis_trades["prior_trend"] = np.where(
    analysis_trades["trend_move_atr"] >= 0,
    "up",
    "down",
)
analysis_trades["volatility_regime"] = pd.qcut(
    analysis_trades["atr_pct"],
    q=3,
    labels=["low", "medium", "high"],
    duplicates="drop",
)
analysis_trades["efficiency_regime"] = pd.qcut(
    analysis_trades["efficiency"],
    q=3,
    labels=["noisy", "mixed", "directional"],
    duplicates="drop",
)

minimum_trades = 20


def regime_metrics(frame: pd.DataFrame) -> pd.Series:
    pnl = frame["pnl_points"]
    wins = pnl[pnl > break_even_threshold]
    losses = pnl[pnl < -break_even_threshold]

    return pd.Series(
        {
            "trades": len(frame),
            "net_points": pnl.sum(),
            "expectancy": pnl.mean(),
            "profit_factor": (
                wins.sum() / abs(losses.sum())
                if not losses.empty
                else np.inf
            ),
            "win_rate_pct": (
                100 * len(wins) / (len(wins) + len(losses))
                if len(wins) + len(losses)
                else 0
            ),
        }
    )


regime_stats = (
    analysis_trades.groupby(
        [
            "entry_year",
            "volatility_regime",
            "efficiency_regime",
            "prior_trend",
        ],
        observed=True,
    )
    .apply(regime_metrics, include_groups=False)
    .reset_index()
)
regime_stats = (
    regime_stats.loc[regime_stats["trades"] >= minimum_trades]
    .sort_values(
        ["entry_year", "profit_factor", "net_points"],
        ascending=[True, False, False],
    )
    .round(2)
)
yearly_feature_summary = (
    analysis_trades.groupby("entry_year", as_index=False)
    .agg(
        trades=("pnl_points", "count"),
        net_points=("pnl_points", "sum"),
        avg_atr_points=("atr_points", "mean"),
        avg_efficiency=("efficiency", "mean"),
        avg_channel_width_atr=("channel_width_atr", "mean"),
    )
    .round(2)
)

print(f"REGIME PERFORMANCE — MINIMUM {minimum_trades} TRADES")
print(regime_stats.to_markdown(index=False))
print("\nYEARLY MARKET CONTEXT AT ENTRY")
print(yearly_feature_summary.to_markdown(index=False))


In [ ]:
# 3. Analyse alignment between trade side and prior trend
analysis_trades["side"] = analysis_trades["side"].str.upper()
analysis_trades["trend_alignment"] = np.where(
    (
        analysis_trades["side"].eq("BUY")
        & analysis_trades["prior_trend"].eq("up")
    )
    | (
        analysis_trades["side"].eq("SELL")
        & analysis_trades["prior_trend"].eq("down")
    ),
    "aligned",
    "counter_trend",
)


def alignment_metrics(frame: pd.DataFrame) -> dict:
    pnl = frame["pnl_points"]
    wins = pnl[pnl > break_even_threshold]
    losses = pnl[pnl < -break_even_threshold]

    return {
        "trades": len(frame),
        "net_points": pnl.sum(),
        "expectancy": pnl.mean(),
        "profit_factor": (
            wins.sum() / abs(losses.sum())
            if not losses.empty
            else np.inf
        ),
        "win_rate_pct": (
            100 * len(wins) / (len(wins) + len(losses))
            if len(wins) + len(losses)
            else 0
        ),
    }


def grouped_metrics(
    frame: pd.DataFrame,
    group_columns: list[str],
) -> pd.DataFrame:
    rows = []

    for keys, group in frame.groupby(group_columns, observed=True):
        if not isinstance(keys, tuple):
            keys = (keys,)

        row = dict(zip(group_columns, keys))
        row.update(alignment_metrics(group))
        rows.append(row)

    return pd.DataFrame(rows)


by_alignment = (
    grouped_metrics(analysis_trades, ["entry_year", "trend_alignment"])
    .sort_values(["entry_year", "profit_factor"], ascending=[True, False])
    .round(2)
)
by_regime_and_alignment = (
    grouped_metrics(
        analysis_trades,
        [
            "entry_year",
            "volatility_regime",
            "efficiency_regime",
            "trend_alignment",
        ],
    )
    .loc[lambda frame: frame["trades"] >= minimum_trades]
    .sort_values(
        ["entry_year", "profit_factor", "net_points"],
        ascending=[True, False, False],
    )
    .round(2)
)
by_side_and_prior_trend = (
    grouped_metrics(analysis_trades, ["entry_year", "side", "prior_trend"])
    .sort_values(
        ["entry_year", "side", "profit_factor"],
        ascending=[True, True, False],
    )
    .round(2)
)

print("ALIGNMENT WITH PRIOR TREND")
print(by_alignment.to_markdown(index=False))
print(f"\nREGIME + ALIGNMENT (minimum {minimum_trades} trades)")
print(by_regime_and_alignment.to_markdown(index=False))
print("\nSIDE + PRIOR TREND")
print(by_side_and_prior_trend.to_markdown(index=False))


In [ ]:
# 4. Compare the baseline against the aligned-only candidate filter
baseline = analysis_trades.copy()
aligned_only = analysis_trades.loc[
    analysis_trades["trend_alignment"].eq("aligned")
].copy()


def performance_by_year(frame: pd.DataFrame, rule: str) -> pd.DataFrame:
    rows = []

    for year, group in frame.groupby("entry_year", sort=True):
        group = group.sort_values("exit_time")
        pnl = group["pnl_points"]
        gross_profit = pnl[pnl > 0].sum()
        gross_loss = -pnl[pnl < 0].sum()
        profit_factor = (
            gross_profit / gross_loss if gross_loss > 0 else np.nan
        )
        equity = pnl.cumsum()
        max_closed_dd = (equity.cummax() - equity).max()

        rows.append(
            {
                "rule": rule,
                "year": int(year),
                "trades": len(group),
                "net_points": pnl.sum(),
                "expectancy": pnl.mean(),
                "profit_factor": profit_factor,
                "max_closed_dd": max_closed_dd,
                "best_trade": pnl.max(),
                "net_without_best": pnl.sum() - pnl.max(),
            }
        )

    return pd.DataFrame(rows)


comparison = pd.concat(
    [
        performance_by_year(baseline, "baseline"),
        performance_by_year(aligned_only, "aligned_only"),
    ],
    ignore_index=True,
).sort_values(["year", "rule"])

baseline_by_year = comparison.loc[
    comparison["rule"].eq("baseline")
].set_index("year")
aligned_by_year = comparison.loc[
    comparison["rule"].eq("aligned_only")
].set_index("year")
delta = pd.DataFrame(
    {
        "trades_removed": (
            baseline_by_year["trades"] - aligned_by_year["trades"]
        ),
        "net_points_change": (
            aligned_by_year["net_points"] - baseline_by_year["net_points"]
        ),
        "drawdown_change": (
            aligned_by_year["max_closed_dd"]
            - baseline_by_year["max_closed_dd"]
        ),
        "pf_change": (
            aligned_by_year["profit_factor"]
            - baseline_by_year["profit_factor"]
        ),
    }
).reset_index()

print("BASELINE VS ALIGNED-ONLY")
print(comparison.round(2).to_markdown(index=False))
print("\nFILTER EFFECT: ALIGNED-ONLY MINUS BASELINE")
print(delta.round(2).to_markdown(index=False))
